# 🌊 Recreating the Dynamic Topography & Geoid Globes

This notebook demonstrates how to create 3D-printable globes of long-wavelength potential fields like dynamic topography and geoid height (Section 3.3 of Koelemeijer & Winterbourne 2021). We'll composite multiple grids together using weighting factors.

### 🌎 Scientific Context
- **Dynamic Topography**: Elevated or depressed regions of the Earth's crust caused by the convective flow of the mantle underneath, pushing land up or pulling it down by $\pm 1\text{ km}$ over vast wavelengths.
- **The Geoid**: The gravitational shape of the Earth, representing anomalies in gravitational attraction due to density variations in the deep Earth.

Because these fields are long-wavelength and subtle, we apply a massive vertical exaggeration (**300×**) and composite them with a coastline step so the continents remain recognizable.

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    LineDisplacer,
    calculate_displacement_scale
)

## Step 2: Load Topography and Generate a Long-Wavelength Field

We load ETOPO topography. Because dynamic topography and geoid datasets are long-wavelength, we simulate a global geoid/dynamic topography grid using spherical harmonic waves for the demo.

*(To use the real EIGEN-6C4 geoid or Hoggard et al. 2016 dynamic topography grids, check the [Data Sourcing Guide](../../user_guide/8_where_to_get_data.md) and replace the paths below).* 

In [ ]:
# 1. Load ETOPO surface topography
full_grid = GeographicGrid.from_netcdf(
    "../../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
grid_ds = GeographicGrid(
    lats=full_grid.lats[::15], lons=full_grid.lons[::15], grid=full_grid.grid[::15, ::15]
)

# 2. Generate a long-wavelength global geoid simulation (placeholder)
LON, LAT = np.meshgrid(grid_ds.lons, grid_ds.lats)
geoid_data = 80.0 * np.sin(2 * np.radians(LON)) * np.cos(2 * np.radians(LAT)) # geoid anomaly in meters
geoid_grid = GeographicGrid(lats=grid_ds.lats, lons=grid_ds.lons, grid=geoid_data)

print("Topography loaded and geoid field simulated successfully.")

## Step 3: Composite the Grids (Topography + Long-Wavelength Field)

We add the grids together with weighting factors to emphasize the geoid field while retaining ocean basins and mountain ranges for reference.

In [ ]:
# Define weights
geoid_weight = 300.0  # High weight because geoid is max ~100m, compared to mountains ~8000m
topo_weight = 1.0

composite_data = (grid_ds.grid * topo_weight) + (geoid_grid.grid * geoid_weight)
composite_grid = GeographicGrid(lats=grid_ds.lats, lons=grid_ds.lons, grid=composite_data)

print("Composited grids successfully.")

## Step 4: Build and Displace the Model

We create a hollow sphere and apply the composite displacement with a 40× exaggeration scale, and then add a 0.8 mm coastline step.

In [ ]:
model_radius_mm = 40.0

# Initialize a hollow model
model = GlobeModel(
    n_points=6000,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=0.7,
)

# Calculate scale factor
scale = calculate_displacement_scale(model_radius_mm, vertical_exagg=40.0, grid_units='m')

# Apply composite displacement
model.outer.displace(GridDisplacer(composite_grid), scale=scale)

# Add coastline step boundary for spatial context
model.outer.displace(LineDisplacer(
    shapefile_path="../../inputs/coastlines/ne_110m_coastline.shp",
    displacement=0.8,
    width_degrees=0.5
))

# Configure magnets and export
model.configure_magnets(
    diameter=5.0, height=2.0, n_magnets=3, position=0.0, add_bosses=True
)

os.makedirs('../../outputs', exist_ok=True)
model.export_hemispheres(
    "../../outputs/paper_geoid_top.stl",
    "../../outputs/paper_geoid_bottom.stl",
    engine='manifold'
)
print("Watertight geoid/dynamic topography hemispheres exported!")